In [1]:
import numpy as np

import mbuild as mb
from mbuild import Compound

# Ignore these imports, this is just to prevent warning messages for the purposes of the workshop
from mbuild import mBuildLogger
import logging
mbuild_logger = mBuildLogger()
mbuild_logger.library_logger.setLevel(logging.ERROR)

In [2]:
# Create molecules

In [3]:
benzene_smiles = mb.load("c1ccccc1", smiles=True)
benzene_file = mb.load("../files/benzene.mol2")

In [4]:
print(type(benzene_smiles))
print(type(benzene_file))

<class 'mbuild.compound.Compound'>
<class 'mbuild.compound.Compound'>


In [5]:
benzene_smiles.visualize().show()
benzene_file.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
class Triangle(Compound):
    def __init__(self, bond_length, A_name="_A", B_name="_B", C_name="_C"):
        super(Triangle, self).__init__()
        bead_A = Compound(name=A_name, pos=(0,0,0))
        bead_B = Compound(name=B_name, pos=(bond_length, 0, 0))
        ypos_C = np.sqrt((bond_length)**2 - (bond_length/2)**2)
        pos_C = (bond_length / 2, ypos_C, 0)
        bead_C = Compound(name=C_name, pos=pos_C)
        self.add([bead_A, bead_B, bead_C])
        self.add_bond([bead_A, bead_B])
        self.add_bond([bead_A, bead_C])
        self.add_bond([bead_B, bead_C])

In [7]:
triangle = Triangle(bond_length=0.2)
triangle.visualize(color_scheme={"_A": "red", "_B": "blue", "_C": "green"})

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
class FattyAcid(Compound):
    SMILES = {
        "cis":   "CCCCCCCC/C=C\\CCCCCCCC(=O)O",   # oleic acid
        "trans": "CCCCCCCC/C=C/CCCCCCCC(=O)O",    # elaidic acid
    }
    def __init__(self, isomer="cis"):
        super(FattyAcid, self).__init__()
        mb.load(self.SMILES[isomer], smiles=True, compound=self)
        self.name = "FA"

In [9]:
fa_cis = FattyAcid("cis")
fa_trans = FattyAcid("trans")

fa_cis.visualize().show()
fa_trans.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
fa_cis = FattyAcid(isomer="cis")
fa_rans = FattyAcid(isomer="trans")
water = mb.load("O", smiles=True)
water.name = "Water"

mixture = mb.packing.fill_box(
    compound=[water, fa_cis, fa_trans],
    n_compounds=[200, 20, 20],
    box=[4, 4, 4],
)
mixture.visualize().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
print(type(mixture))

<class 'mbuild.compound.Compound'>


In [12]:
for child in mixture.children[:5]:
    print(child.name, child.pos, child.mass, type(child))

Water [2.9469662  3.6759519  3.10368817] 18.015 <class 'mbuild.compound.Compound'>
Water [3.06751657 3.24739717 2.60386267] 18.015 <class 'mbuild.compound.Compound'>
Water [0.5969922  0.673453   2.06560247] 18.015 <class 'mbuild.compound.Compound'>
Water [1.9014693  3.17342803 3.46622687] 18.015 <class 'mbuild.compound.Compound'>
Water [2.37856877 1.0091798  3.3025683 ] 18.015 <class 'mbuild.compound.Compound'>


In [13]:
molecule = mixture.children[0]
oxygen = molecule.children[0]
h1 = molecule.children[1]
h2 = molecule.children[2]

In [14]:
molecule.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## General Molecular Simulation Object (GMSO)

- Graph-based atom typing: SMARTS string `[C;X3]([O;X1])(C)(C)(H)` → Molecular Graph (networkX)
- Uses an XML file scheme to define:
    - Analytical forms of forces and parameters: $E(r) = 4\epsilon\left[\left(\dfrac{\sigma}{r}\right)^{12} - \left(\dfrac{\sigma}{r}\right)^6\right]$
    - Parameter values and units: $\sigma=0.32 \, \textit{nm}; \epsilon=0.45 \frac{kJ}{mol}$
    - Atom type definitions: `[C;X3]([O;X1])([O;X2])(C)`
    - Atom types and Atom classes: `ca` 

In [15]:
import gmso
from gmso.core.forcefield import ForceField
from gmso.parameterization.parameterize import apply

In [16]:
gaff = ForceField("../files/gaff.xml") # General Amber forcefield
spcfw = ForceField("../files/spcfw.xml") # A non-rigid water model

In [17]:
mixture_topology = mixture.to_gmso() # `mixture` is an mBuild compound, this converts it to a gmso topology, so we have a new python object

In [18]:
print(type(mixture_topology))

<class 'gmso.core.topology.Topology'>


In [19]:
apply(
    top=mixture_topology,
    identify_connections=True,
    forcefields={"FA": gaff, "Water": spcfw},
    speedup_by_moltag=True
)

<Topology Topology, 2760 sites,
 12400 connections,
 15160 potentials,
 id: 13164141392>

In [20]:
for particle in [p for p in mixture.particles()][:5]:
    print(particle.pos, particle.name)

[2.9243752 3.6601934 3.1332489] O
[3.023018  3.6586034 3.1249982] H
[2.8935054 3.7090589 3.0528174] H
[3.1016288 3.2533438 2.6246836] O
[3.0699715 3.3026971 2.5449148] H


In [21]:
print("=" * 10, "Atom Types", "=" * 10)
for site in mixture_topology.sites[:3]:
    print(site.atom_type.name, site.atom_type.parameters, site.atom_type.charge, site.atom_type.expression)
for site in mixture_topology.sites[605:608]:
    print(site.atom_type.name, site.atom_type.parameters, site.atom_type.expression)
print()

print("=" * 10, "Bond Types", "=" * 10)
bond = mixture_topology.bonds[0]
print(f"({bond.connection_members[0].atom_type.name}-{bond.connection_members[1].atom_type.name})", bond.bond_type.parameters, bond.bond_type.expression)

bond = mixture_topology.bonds[500]
print(f"({bond.connection_members[0].atom_type.name}-{bond.connection_members[1].atom_type.name})", bond.bond_type.parameters, bond.bond_type.expression)
print()

print("=" * 10, "Angle Types", "=" * 10)
angle = mixture_topology.angles[500]
print(f"({angle.connection_members[0].atom_type.name}-{angle.connection_members[1].atom_type.name}-{angle.connection_members[2].atom_type.name})", angle.angle_type.parameters, angle.angle_type.expression)

========== Atom Types ==========
OW {'epsilon': unyt_quantity(0.649, 'kJ/mol'), 'sigma': unyt_quantity(0.3166, 'nm')} -0.82 elementary_charge 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0., 'nm')} 0.41 elementary_charge 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
HW {'epsilon': unyt_quantity(0., 'kJ/mol'), 'sigma': unyt_quantity(0., 'nm')} 0.41 elementary_charge 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
c3 {'epsilon': unyt_quantity(0.4577296, 'kJ/mol'), 'sigma': unyt_quantity(0.33996695, 'nm')} 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
c3 {'epsilon': unyt_quantity(0.4577296, 'kJ/mol'), 'sigma': unyt_quantity(0.33996695, 'nm')} 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
c3 {'epsilon': unyt_quantity(0.4577296, 'kJ/mol'), 'sigma': unyt_quantity(0.33996695, 'nm')} 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)

========== Bond Types ==========
(OW-HW) {'k': unyt_quantity(443153.38, 'kJ/(mol*nm**2)'), 'r_eq': unyt_qu

In [22]:
from gmso.external.convert_hoomd import to_gsd_snapshot, to_hoomd_forcefield
from gmso.formats import write_lammpsdata, write_gro

In [23]:
hoomd_snapshot, _ = gmso.external.convert_hoomd.to_gsd_snapshot(top=mixture_topology)
hoomd_forces, _ = gmso.external.to_hoomd_forcefield(top=mixture_topology, r_cut=1.3)

2026-06-26 13:00:16,457 - gmso.external.convert_hoomd - WARNING - Neither base_units or auto_scale is provided, so default units of {'length': unyt_quantity(1, 'nm'), 'energy': unyt_quantity(1, 'kJ/mol'), 'mass': unyt_quantity(1, 'amu')} are inferred.


In [24]:
write_lammpsdata(top=mixture_topology, filename="ffa_mixture.data")

In [25]:
!head -n 40 ffa_mixture.data

Topology written by cj4006 at 2026-06-26 13:00:26.122944 using the GMSO LAMMPS Writer


2760 atoms
2520 bonds
4200 angles
6920 dihedrals
40 impropers

10 atom types
10 bond types
15 angle types
28 dihedral types
1 improper types

0.000000 40.000000 xlo xhi
0.000000 40.000000 ylo yhi
0.000000 40.000000 zlo zhi
0.000000 0.000000 0.000000 xy xz yz

Masses
#	mass (amu)
1	0.000	# HW
2	0.000	# OW
3	12.010	# c
4	12.010	# c2
5	12.010	# c3
6	1.008	# ha
7	1.008	# hc
8	1.008	# ho
9	16.000	# o
10	16.000	# oh

Pair Coeffs # 4*epsilon*(-sigma**6/r**6 + sigma**12/r**12)
#	epsilon (kcal/mol)	sigma (Å)
1	0.00000		0.00000		# HW
2	0.15511		3.16600		# OW
3	0.08600		3.39967		# c
4	0.08600		3.39967		# c2
5	0.10940		3.39967		# c3


In [26]:
write_gro(top=mixture_topology, filename="ffa_mixture.gro")

In [27]:
!head -n 25 ffa_mixture.gro

Topology written by GMSO 0.16.2 at 2026-06-26 13:00:27.273904
2760
    1Water    O    1   2.924   3.660   3.133
    1Water    H    2   3.023   3.659   3.125
    1Water    H    3   2.894   3.709   3.053
    2Water    O    4   3.102   3.253   2.625
    2Water    H    5   3.070   3.303   2.545
    2Water    H    6   3.031   3.186   2.642
    3Water    O    7   0.632   0.663   2.082
    3Water    H    8   0.596   0.636   1.994
    3Water    H    9   0.563   0.722   2.121
    4Water    O   10   1.916   3.182   3.503
    4Water    H   11   1.864   3.102   3.476
    4Water    H   12   1.925   3.236   3.420
    5Water    O   13   2.348   1.004   3.329
    5Water    H   14   2.366   0.953   3.246
    5Water    H   15   2.421   1.071   3.333
    6Water    O   16   1.444   2.090   2.023
    6Water    H   17   1.507   2.013   2.021
    6Water    H   18   1.363   2.056   1.978
    7Water    O   19   1.658   0.768   1.478
    7Water    H   20   1.726   0.759   1.549
    7Water    H   21   1.586   0.

# Recipe Construction

In [28]:
class FattyAcidMixture(Compound):
    def __init__(
        self,
        n_fatty_acids: int,
        cis_ratio: float,
        solvent: mb.Compound,
        n_solvents: int,
        box_length: float
    ):
        super(FattyAcidMixture, self).__init__()
        # ADD THE RECIPE BUILDER SCRIPT HERE; LOOK ABOVE FOR HINTS

In [29]:
# Test your recipe here after filling in the cell above
water = mb.load("O", smiles=True)
water.name = "Water"

mixture = FattyAcidMixture(
    n_fatty_acids=100,
    cis_ratio=0.64,
    solvent=water, # Pass in an instance of an mBuild Compound (mb.load)
    n_solvents=200, 
    box_length=5
)

In [30]:
# Try with a different solvent, like hexane (mb.load("CCCCCC", smiles=True))

In [31]:
# The cell below contains one implementation if you want ideas/help

In [32]:
class FattyAcidMixture(Compound):
    def __init__(
        self,
        n_fatty_acids: int,
        cis_ratio: float,
        solvent: mb.Compound,
        n_solvents: int,
        box_length: float
    ):
        super(FattyAcidMixture, self).__init__()
        # Use the FattyAcid class we already made above.
        # Other options: Directly use the corresponding SMILES strings with mb.load("smiles-string", smiles=True) here
        # store the cis and trans files, and load them here mb.load("path-to-file")
        cis_fa = FattyAcid("cis") 
        trans_fa = FattyAcid("trans")
        # Solve for the remaining information needed by mbuild.fill_box()
        n_cis = int(n_fatty_acids * cis_ratio)
        n_trans = int(n_fatty_acids - n_cis)
        self.add(
           mb.fill_box(
            compound=[solvent, cis_fa, trans_fa],
            n_compounds=[n_solvents, n_cis, n_trans],
            box=[box_length, box_length, box_length]
            )
        )
        self.name = f"mixture_{cis_ratio}_cis_ratio"

In [33]:
# Because of the way we designed the recipe, we can easily use it with different solvent molecules:

In [34]:
mixture = FattyAcidMixture(n_fatty_acids=100, cis_ratio=0.45, solvent=water, n_solvents=230, box_length=6)
mixture.visualize()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
hexane = mb.load("CCCCCC", smiles=True)
mixture = FattyAcidMixture(n_fatty_acids=100, cis_ratio=0.56, solvent=hexane, n_solvents=230, box_length=6)
mixture.visualize()

In [ ]:
dmso = mb.load("CS(=O)C", smiles=True)
mixture = FattyAcidMixture(n_fatty_acids=100, cis_ratio=0.45, solvent=dmso, n_solvents=230, box_length=6)
mixture.visualize()